In [3]:
%%time
import os

# Set this BEFORE importing torch/transformers in a fresh kernel
os.environ["CUDA_VISIBLE_DEVICES"] = "6,7"

import torch
from transformers import AutoProcessor, Gemma3ForConditionalGeneration

if not torch.cuda.is_available():
    raise RuntimeError("CUDA is not available.")

print("Visible GPUs:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(f"cuda:{i}: {torch.cuda.get_device_name(i)}")

model_name = "google/gemma-3-4b-it"

max_memory = {
    0: "12GiB",
    1: "12GiB",
    "cpu": "100GiB",
}

model = Gemma3ForConditionalGeneration.from_pretrained(
    model_name,
    torch_dtype=torch.bfloat16,
    device_map="balanced",
    max_memory=max_memory,
    low_cpu_mem_usage=True,
).eval()

processor = AutoProcessor.from_pretrained(model_name)

print("\nModel device map:")
print(model.hf_device_map)

messages = [
    {
        "role": "system",
        "content": [
            {"type": "text", "text": "You are a helpful assistant."}
        ],
    },
    {
        "role": "user",
        "content": [
            {"type": "text", "text": "Give me a short introduction to large language models."}
        ],
    },
]

inputs = processor.apply_chat_template(
    messages,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt",
).to(model.device, dtype=torch.bfloat16)

input_len = inputs["input_ids"].shape[-1]

print("\nInput device:", inputs["input_ids"].device)
print("Prompt token count:", input_len)

with torch.inference_mode():
    generation = model.generate(
        **inputs,
        max_new_tokens=256,
        do_sample=False,
        use_cache=True,
    )

new_tokens = generation[0][input_len:]

print("Generated token count:", new_tokens.shape[0])
print("First generated IDs:", new_tokens[:20].tolist())

response = processor.decode(new_tokens, skip_special_tokens=True).strip()

# print("\nResponse repr:", repr(response))
print("\nResponse:\n" + (response if response else "[EMPTY RESPONSE]"))

Visible GPUs: 2
cuda:0: NVIDIA A16
cuda:1: NVIDIA A16


Loading weights: 100%|██████████| 883/883 [00:02<00:00, 439.08it/s]



Model device map:
{'model.vision_tower': 0, 'model.multi_modal_projector': 0, 'model.language_model.embed_tokens': 0, 'lm_head': 0, 'model.language_model.layers.0': 0, 'model.language_model.layers.1': 0, 'model.language_model.layers.2': 0, 'model.language_model.layers.3': 0, 'model.language_model.layers.4': 0, 'model.language_model.layers.5': 0, 'model.language_model.layers.6': 0, 'model.language_model.layers.7': 0, 'model.language_model.layers.8': 0, 'model.language_model.layers.9': 0, 'model.language_model.layers.10': 0, 'model.language_model.layers.11': 1, 'model.language_model.layers.12': 1, 'model.language_model.layers.13': 1, 'model.language_model.layers.14': 1, 'model.language_model.layers.15': 1, 'model.language_model.layers.16': 1, 'model.language_model.layers.17': 1, 'model.language_model.layers.18': 1, 'model.language_model.layers.19': 1, 'model.language_model.layers.20': 1, 'model.language_model.layers.21': 1, 'model.language_model.layers.22': 1, 'model.language_model.laye

In [2]:
new_tokens

tensor([ 19058, 236764,   1590, 236789, 236751,    496,   2822,  12650,    531,
         25093,  22160,  40121,    568,   2182,  21706,   1473,    108,   1018,
         31534,  22160,  40121,    568,   2182,  21706, 236768,    659,  13981,
          2126, 236764,   2126,   6538,   5194,   6607,    600,    740,   3050,
           532,   8729,   3246, 236772,   5282,   1816,  99382, 236743,    108,
          8291, 236789, 236751,    496,  25890,    529,    506,   2307,   2432,
           531,   1281, 236787,    108, 236829,   5213,   7634, 236789,    500,
          5284,    580,  45556,  28243,    568,  12553,   1473,   1018,  39799,
        236764,    901,   1161,    496,   1722,    529,  12498,   2760,    999,
         28680,   4735,   1827,    107, 236829,   5213, 236913,  31534, 236919,
          2820,  12566,   1262,  53121,   2195, 236858,    500,  15453,    580,
           808, 109693, 236829,  12136,    529,   1816,   1262,   1271,   1751,
           506,   4251,   8379, 236764, 

In [ ]:
def gpu_memory(device_id):
    allocated = torch.cuda.memory_allocated(device_id) / 1024**3
    reserved = torch.cuda.memory_reserved(device_id) / 1024**3
    total = torch.cuda.get_device_properties(device_id).total_memory / 1024**3
    print(
        f"cuda:{device_id} | "
        f"allocated: {allocated:.2f} GiB | "
        f"reserved: {reserved:.2f} GiB | "
        f"total: {total:.2f} GiB"
    )

gpu_memory(0)
gpu_memory(1)

In [ ]:
import gc
import torch

# Remove Python references to GPU-resident objects
for name in (
    "model",
    "tokenizer",
    "model_inputs",
    "generated_ids",
    "response",
    "text",
    "messages",
):
    globals().pop(name, None)

# Force Python garbage collection, then release unused PyTorch cache
gc.collect()

for device_id in range(torch.cuda.device_count()):
    with torch.cuda.device(device_id):
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()